# Session 2: Creating APIs with FastAPI

**Learning goals:**

- Build and test FastAPI endpoints that return JSON.
- Validate inputs and define response models.
- Wrap weather and arXiv APIs with your own endpoints.

**Prerequisites:**

- Complete Session 1 and know basic Python functions.
- Have Python 3.12+, a notebook environment, a terminal and internet access; install the Section 2 packages.
- Have a WeatherAPI account and API key for live weather examples.


## 1. Client/Server architecture

A client initiates an HTTP request. A server listens for requests and returns HTTP responses. FastAPI defines the application behaviour.

FastAPI is a server when responding to our Python client, and a client when requesting data from WeatherAPI. The API key stays on the FastAPI server; callers of our local API do not need that key.

![A Python client requests weather from FastAPI. FastAPI requests detailed JSON from WeatherAPI, shapes the data and returns simpler JSON. The API key stays on the FastAPI server.](assets/session2/client-server-flow.svg)


## 2. Your first FastAPI application

### Prepare the environment

On a TAFE Windows computer, use this setup file:

[Download the Windows setup script (setup_api_ss.bat)](assets/session2/setup_api_ss.bat)

1. Save the file to **Downloads**, keeping the `.bat` extension.
2. Double-click `setup_api_ss.bat`. Python 3.12 or later must already be installed, and the computer needs internet access.
3. Wait for **Setup complete**. The script creates `api_ss` inside `C:\ProgramData\Laragon\www`, creates and activates `.venv` inside `api_ss`, then installs the course libraries. After these steps succeed, it defines `API_SS_PATH` as `C:\ProgramData\Laragon\www\api_ss`.
4. Keep the Command Prompt opened by the script. The environment is active, and `%API_SS_PATH%` refers to your course folder. Use this window for the server commands below.
5. Open the `api_ss` folder in VS Code. Use **Python: Select Interpreter** to choose `.venv\Scripts\python.exe`; choose the same environment in the notebook's **Select Kernel** menu. VS Code needs the Python and Jupyter extensions.

You can rerun the setup file to finish an interrupted installation. It reuses an existing working `.venv`. If setup fails, keep the error window open and ask your lecturer for help.

`API_SS_PATH` is available in the setup window and programs started from it. If you close that window, open a new **Command Prompt** and restore the path and environment:

```bat
set "API_SS_PATH=C:\ProgramData\Laragon\www\api_ss"
cd /d "%API_SS_PATH%"
call .venv\Scripts\activate.bat
```

If you already have a Python environment on another computer, install the same packages in the environment used by your notebook and terminal:

```bash
python -m pip install fastapi uvicorn requests httpx ipykernel pandas arxiv
```

### Define two routes

`FastAPI()` creates the application. A decorator such as `@app.get("/")` connects an HTTP method and path to a Python function. Returning a dictionary produces a JSON response.

The `/weather` route below uses fixed demonstration values, not live observations. `city: str` declares a required query parameter.


In [ ]:
from fastapi import FastAPI

# Give the application a title for its interactive documentation.
app = FastAPI(title="Demonstration Weather API")


@app.get("/")
def root():
    """Return a welcome message."""
    return {"message": "Hello, World!"}


@app.get("/weather")
def demo_weather(city: str):
    """Return fixed sample weather for a requested city."""
    return {"city": city, "temp_c": 25.0, "condition": "Sunny"}


### Run the server and open its documentation

1. [Download stage2_demo.py](assets/session2/stage2_demo.py) and save it in **Downloads**. Alternatively, copy the preceding application code cell into a new file and save it as `stage2_demo.py` in **Downloads**. Keep the `.py` extension.
2. In the Command Prompt left open by the setup script, copy the Python file into the course folder and start Uvicorn:

```bat
copy "%USERPROFILE%\Downloads\stage2_demo.py" "%API_SS_PATH%\stage2_demo.py"
cd /d "%API_SS_PATH%"
python -m uvicorn stage2_demo:app --reload
```

If you saved the file somewhere else, change the source path in `copy`. The destination remains `%API_SS_PATH%`.

`python -m uvicorn` runs Uvicorn from the active environment. `stage2_demo` identifies `stage2_demo.py` in the current folder, and `app` is the application object inside it. `--reload` restarts the development server when that Python file changes. Notebook edits do not update the saved `.py` file.

Open [the welcome endpoint](http://127.0.0.1:8000/) and [Swagger UI](http://127.0.0.1:8000/docs). Expand `GET /weather`, select **Try it out**, enter `Perth`, then select **Execute**. Compare the request URL, status code and JSON response. Open `/openapi.json` to inspect the machine-readable API description used by the documentation.

`127.0.0.1` means this computer and `8000` is the listening port. Keep the terminal running while using the client below. `--reload` is for local development.

[FastAPI first steps](https://fastapi.tiangolo.com/tutorial/first-steps/)


In [ ]:
import requests

LOCAL_API_URL = "http://127.0.0.1:8000"

# params encodes the query string; timeout limits how long the client waits.
demo_response = requests.get(
    f"{LOCAL_API_URL}/weather",
    params={"city": "Perth"},
    timeout=10,
)
demo_response.raise_for_status()
print("Response:", demo_response.json())


## 3. Wrap a real API: a weather microservice

### Configure WeatherAPI

Here we wrap the same current-weather endpoint with our own FastAPI route. 

[WeatherAPI documentation](https://www.weatherapi.com/docs/) · [Current weather fields](https://www.weatherapi.com/docs/realtime-api/)

### Request and shape the data

This stage creates a new application to replace the fixed-data prototype. 

In `/weather/`, `city` does not appear in the route path. FastAPI treats `city: str` as a required query parameter, so `/weather/?city=Perth` passes `Perth` to the handler. Section 5 adds forecast and historical weather routes.


In [ ]:
import requests
from fastapi import FastAPI

# Replace the placeholder with your WeatherAPI key.
WEATHER_API_KEY = "YOUR_WEATHERAPI_KEY"
CURRENT_WEATHER_URL = "https://api.weatherapi.com/v1/current.json"

# Replace the fixed-data prototype with a live weather application.
app = FastAPI(title="Live Weather API")

@app.get("/")
def root():
    """Describe the local service."""
    return {"message": "Weather API", "docs": "/docs"}

@app.get("/weather/")
def get_weather(city: str):
    """Return simplified weather for a city."""
    # key authenticates the request; q selects a location; aqi omits air-quality data.
    params = {"key": WEATHER_API_KEY, "q": city.strip(), "aqi": "no"}
    response = requests.get(
        CURRENT_WEATHER_URL,
        params=params,
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()

    # Flatten the location and current objects into one small record.
    return {
        "city": data["location"]["name"],
        "current_temp": data["current"]["temp_c"],
        "condition": data["current"]["condition"]["text"],
    }


Stop the previous server with **Ctrl+C**. [Download stage3_weather.py](assets/session2/stage3_weather.py), saving the complete file in **Downloads**. Open the file and replace `"YOUR_WEATHERAPI_KEY"` with your key, then save it. In the same Command Prompt, where `.venv` is active, copy the file into the course folder and run it:

```bat
copy "%USERPROFILE%\Downloads\stage3_weather.py" "%API_SS_PATH%\stage3_weather.py"
cd /d "%API_SS_PATH%"
python -m uvicorn stage3_weather:app --reload
```

In `/docs`, try `/weather/?city=Perth`. Compare it with the original WeatherAPI response: the service exposes a small contract instead of forwarding every upstream field. This is **data shaping**.

**Try it:** Request another Australian city and compare the returned values. If a request fails, check the server terminal, your API key and the WeatherAPI documentation.


## 4. Pydantic response models

### Define the response contract

Type hints describe values. Pydantic models describe structured data. FastAPI uses the response model to validate and serialise returned data, filter undeclared fields and document the response.

`WeatherResponse` describes the city, current temperature and weather condition. `Field` adds descriptions to the documentation. Temperature fields are in degrees Celsius.

The next cell defines the response model and attaches it to `get_weather()` using `response_model=WeatherResponse`. The city uses a plain `str` type hint.

Run the cell to replace the previous application with this version. Section 5 adds more routes to the same application.


In [ ]:
from pydantic import BaseModel, Field


class WeatherResponse(BaseModel):
    city: str = Field(description="Resolved location name")
    current_temp: float = Field(description="Current temperature in degrees Celsius")
    condition: str = Field(description="Current weather description")

# Start the final application; Section 5 adds routes to this same object.
app = FastAPI(title="Weather and Research API", version="1.0.0")


@app.get("/")
def root():
    """Describe the service and link to its documentation."""
    return {"message": "Weather and Research API", "docs": "/docs"}


@app.get("/weather/", response_model=WeatherResponse)
def get_weather(
    city: str,
):
    """Retrieve current weather conditions."""
    # key authenticates the request; q selects a location; aqi omits air-quality data.
    params = {"key": WEATHER_API_KEY, "q": city.strip(), "aqi": "no"}
    response = requests.get(
        CURRENT_WEATHER_URL,
        params=params,
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()

    # Flatten the location and current objects into one small record.
    return {
        "city": data["location"]["name"],
        "current_temp": data["current"]["temp_c"],
        "condition": data["current"]["condition"]["text"],
    }


[Download the Section 4 main.py application](assets/session2/main.py) and save it in **Downloads**. Replace `"YOUR_WEATHERAPI_KEY"` in that file with your key and save it. Stop the previous server with **Ctrl+C**. In the same Command Prompt, copy the complete file into the course folder and start it:

```bat
copy "%USERPROFILE%\Downloads\main.py" "%API_SS_PATH%\main.py"
cd /d "%API_SS_PATH%"
python -m uvicorn main:app --reload
```

This file contains the current-weather function from Section 4. In Section 5, you will paste additional function code into this file yourself. In `/docs`, expand **Schemas** to see `WeatherResponse`.

The response model checks and documents the data returned by our server. Pydantic does not fetch data or check whether a real city exists. Section 5 introduces query parameter validation.

[FastAPI response models](https://fastapi.tiangolo.com/tutorial/response-model/)

### Inspect the response schema without a running server

`TestClient` calls the notebook application in the same process. Requesting `/openapi.json` reads the API description without fetching weather, so it needs neither an API key nor internet access.


In [ ]:
from fastapi.testclient import TestClient

# Inspect the API description without starting Uvicorn or requesting live weather.
with TestClient(app) as schema_client:
    schema = schema_client.get("/openapi.json").json()
    print("Weather response schema:", schema["components"]["schemas"]["WeatherResponse"])
    print("Documented paths:", list(schema["paths"]))


**Try it:** Find `WeatherResponse` in `/docs` under **Schemas**. Match each model field to a value returned by `get_weather()`.

**MCP connection:** A tool also needs a clear name, description and parameter schema. These definitions help an AI assistant choose a capability and supply the right arguments. An HTTP route is not automatically an MCP Tool; we will build that connection later.


## 5. Add more functions

Copy each function code cell below to the end of your saved `main.py`, in order. Save the file after each addition; Uvicorn reloads it for testing in `/docs`.


### Update a CSV file with POST

`POST /weather/` uses Pandas to read `weather_records.csv` in the server's working folder. It searches for the same `city` and `date`: update the row if found, otherwise add a row. The first request creates the file and column names.

In `/docs`, try `{"city": "Perth", "date": "2026-10-19", "temp_c": 25, "condition": "Sunny"}`. Submit it again with a different temperature and check the CSV. The response says whether the record was added or updated.

If you created the earlier three-column CSV, rename it before testing this version.


In [ ]:
from datetime import date

import pandas as pd
from pydantic import BaseModel

WEATHER_CSV = "weather_records.csv"
WEATHER_COLUMNS = ["city", "date", "temp_c", "condition"]


class WeatherRecord(BaseModel):
    city: str
    date: date
    temp_c: float
    condition: str


@app.post("/weather/")
def post_weather(weather: WeatherRecord):
    """Read the CSV, find a city and date, then add or update its record."""
    # a+ creates a missing file; write column names if the file is empty.
    with open(WEATHER_CSV, "a+", encoding="utf-8", newline="") as csv_file:
        csv_file.seek(0)
        if csv_file.read(1) == "":
            pd.DataFrame(columns=WEATHER_COLUMNS).to_csv(csv_file, index=False)

    # Keep city and date as text; preserve values such as the city name "NA".
    data = pd.read_csv(
        WEATHER_CSV, dtype={"city": str, "date": str}, keep_default_na=False,
    )
    record = weather.model_dump(mode="json")
    matching = (data["city"] == record["city"]) & (data["date"] == record["date"])

    if matching.any():
        # loc updates the temperature and condition for matching rows.
        data.loc[matching, ["temp_c", "condition"]] = [weather.temp_c, weather.condition]
        action = "updated"
    else:
        # The next row index adds a new record without changing existing rows.
        data.loc[len(data)] = record
        action = "added"

    # index=False saves only the four data columns.
    data.to_csv(WEATHER_CSV, index=False)
    return {
        "message": f"Weather record {action}.",
        "city": weather.city,
        "date": record["date"],
        "file": WEATHER_CSV,
    }


### Get a forecast

`/forecast?city=Perth&days=3` returns daily maximum and minimum temperatures. `days` accepts 1 to 3; available data depends on your WeatherAPI plan. The helper below keeps only the daily fields we need.

[WeatherAPI forecast documentation](https://www.weatherapi.com/docs/forecast-api/)


In [ ]:
import requests
from fastapi import Query
from pydantic import BaseModel, Field


class WeatherDay(BaseModel):
    date: str = Field(description="Weather date in YYYY-MM-DD format")
    max: float = Field(description="Maximum temperature in degrees Celsius")
    min: float = Field(description="Minimum temperature in degrees Celsius")


class DailyWeatherResponse(BaseModel):
    city: str = Field(description="Resolved location name")
    days: list[WeatherDay] = Field(description="Daily forecast or historical records")


FORECAST_WEATHER_URL = "https://api.weatherapi.com/v1/forecast.json"


def shape_daily_weather(data: dict):
    """Keep the daily fields shared by forecast and history responses."""
    return {
        "city": data["location"]["name"],
        "days": [
            {"date": day["date"], "max": day["day"]["maxtemp_c"], "min": day["day"]["mintemp_c"]}
            for day in data["forecast"]["forecastday"]
        ],
    }


@app.get("/forecast", response_model=DailyWeatherResponse)
def get_forecast(
    city: str,
    days: int = Query(1, ge=1, le=3, description="Forecast days; subject to provider plan"),
):
    """Return a daily forecast for a city."""
    # days sets the forecast horizon; q selects a location.
    params = {"key": WEATHER_API_KEY, "q": city.strip(), "days": days}
    response = requests.get(
        FORECAST_WEATHER_URL,
        params=params,
        timeout=10,
    )
    response.raise_for_status()
    return shape_daily_weather(response.json())


### Get historical weather

`/history?city=Perth&dt=YYYY-MM-DD` returns weather for a past day supported by your plan. FastAPI reads `dt` as a date. Reuse `shape_daily_weather()` to format the result.

[WeatherAPI history documentation](https://www.weatherapi.com/docs/history-api/)


In [ ]:
from datetime import date

import requests
from fastapi import Query

HISTORY_WEATHER_URL = "https://api.weatherapi.com/v1/history.json"


@app.get("/history", response_model=DailyWeatherResponse)
def get_history(
    city: str,
    dt: date = Query(..., description="Past date as YYYY-MM-DD; subject to provider plan"),
):
    """Return daily historical weather for a city and date."""
    # dt selects the historical date; isoformat produces YYYY-MM-DD.
    params = {"key": WEATHER_API_KEY, "q": city.strip(), "dt": dt.isoformat()}
    response = requests.get(
        HISTORY_WEATHER_URL,
        params=params,
        timeout=10,
    )
    response.raise_for_status()
    return shape_daily_weather(response.json())


### Search arXiv

Use the `arxiv` library to search for papers and return their titles, summaries and links. `max_results` limits the number of papers.

Install the library in your active terminal with `python -m pip install arxiv`.

[arxiv library documentation](https://lukasschwab.me/arxiv.py/)


In [ ]:
import arxiv
from fastapi import Query
from pydantic import BaseModel

# page_size limits each batch; delay_seconds spaces requests to arXiv.
arxiv_client = arxiv.Client(page_size=10, delay_seconds=3)


class PaperResponse(BaseModel):
    title: str
    summary: str
    source_url: str


@app.get("/arxiv/search", response_model=list[PaperResponse])
def search_papers(
    query: str = Query(..., min_length=1, description="Search topic"),
    max_results: int = Query(5, ge=1, le=10, description="Maximum papers to return"),
):
    """Search arXiv and return a short record for each paper."""
    # all searches across fields; max_results limits the total returned records.
    search = arxiv.Search(query=f"all:{query.strip()}", max_results=max_results)
    return [
        {"title": paper.title, "summary": paper.summary, "source_url": paper.entry_id}
        for paper in arxiv_client.results(search)
    ]


### Call the functions

After adding the functions to `main.py`, keep Uvicorn running and enter your WeatherAPI key in the server file. Run the following client cell in the notebook to call the functions and save a weather record to the CSV.


In [ ]:
from datetime import date, timedelta

import requests

LOCAL_API_URL = "http://127.0.0.1:8000"

# Use a recent past date; available history depends on the provider plan.
HISTORY_EXAMPLE_DATE = (date.today() - timedelta(days=1)).isoformat()
client_examples = [
    ("/weather/", {"city": "Perth"}),
    ("/forecast", {"city": "Perth", "days": 3}),
    ("/history", {"city": "Perth", "dt": HISTORY_EXAMPLE_DATE}),
    ("/arxiv/search", {"query": "machine learning", "max_results": 2}),
]
for route, parameters in client_examples:
    # Allow time for the upstream request and arXiv's request spacing.
    result = requests.get(
        f"{LOCAL_API_URL}{route}", params=parameters,
        timeout=18,
    )
    result.raise_for_status()
    print(result.json())
# json sends the weather record in the request body to update the CSV.
post_response = requests.post(
    f"{LOCAL_API_URL}/weather/",
    json={"city": "Perth", "date": HISTORY_EXAMPLE_DATE, "temp_c": 25, "condition": "Sunny"},
    timeout=18,
)
post_response.raise_for_status()
print("POST:", post_response.json())


**Try it:** POST records for two cities and two dates, then update one record. Check that the CSV keeps the other records without duplicate city-and-date pairs. Request a forecast and search arXiv for another topic.

Edit the saved Python file in `%API_SS_PATH%` to change the running service. `--reload` restarts it after you save.
